# Topic 26 - The Retrieval Pipeline End to End

Companion notebook for `pipeline.py`. One document walks through ingest -> chunk -> embed -> store -> query -> retrieve -> assemble -> answer. The embedder and answerer are deterministic offline stand-ins; the **contracts** (idempotent upserts, batching, reindex triggers, silent failures) are real and pinned by tests.

## Learning objectives

- run the topic's mechanics offline, cell by cell
- connect every demo to the pytest suite that pins it
- leave with numbers you can quote, not impressions


## 0. Imports and the sample corpus

Three policy documents. `max_chars=200` keeps each paragraph its own chunk, which makes the diffing demos easy to see.

In [1]:
from pipeline import (
    SAMPLE_DOCUMENTS, STAGES, Document, Pipeline,
    chunk_document, silent_failure_notes,
)

sample = {doc.doc_id: doc for doc in SAMPLE_DOCUMENTS}
pipeline = Pipeline(max_chars=200, batch_size=2)
print('documents:', sorted(sample))
print('chunks per document:', {d.doc_id: len(chunk_document(d, 200)) for d in SAMPLE_DOCUMENTS})

documents: ['returns', 'shipping', 'warranty']
chunks per document: {'returns': 3, 'shipping': 2, 'warranty': 2}


## 1. First ingest: chunk + embed + store

Each report is auditable: how many chunks, how many were embedded, how many batch requests it took, and any warnings that would otherwise be silent.

In [2]:
for document in SAMPLE_DOCUMENTS:
    report = pipeline.ingest(document)
    print(f"{document.doc_id:<9} reason={report.reason:<12} chunks={report.chunks_total} "
          f"inserted={report.inserted} batches={report.embed_batches}")
    print('  warnings:', report.warnings or ['none'])

print('\ntotal rows in store:', len(pipeline.store))

returns   reason=new_document chunks=3 inserted=3 batches=2
  warnings: ['none']
shipping  reason=new_document chunks=2 inserted=2 batches=1
  warnings: ['none']
warranty  reason=new_document chunks=2 inserted=2 batches=1
  warnings: ['none']

total rows in store: 7


## 2. Re-ingest unchanged: zero embedding calls

This is the operational definition of idempotency. The second ingest compares the manifest and each chunk hash; nothing is embedded and no row is rewritten.

In [3]:
calls_before = pipeline.embedder.calls
report = pipeline.ingest(sample['returns'])
print(report.reason, '| embedded =', report.embedded, '| unchanged =', report.unchanged)
print('batch calls made:', pipeline.embedder.calls - calls_before)
assert report.reason == 'unchanged'
assert report.embedded == 0
assert report.unchanged == report.chunks_total
assert pipeline.embedder.calls == calls_before

unchanged | embedded = 0 | unchanged = 3
batch calls made: 0


## 3. Edit one paragraph: re-embed exactly one chunk

Chunk IDs are `doc_id#ordinal` and each chunk carries a content hash. The hash diff tells the pipeline which row changed. The other two chunks are untouched.

In [4]:
edited = Document('returns', 'Return Policy', sample['returns'].text.replace('30 days', '45 days'))
report = pipeline.ingest(edited)
print(report.reason, '| updated =', report.updated, '| unchanged =', report.unchanged,
      '| embedded =', report.embedded)
assert report.updated == 1
assert report.embedded == 1
stored = {row.chunk_id: row for row in pipeline.store.rows_for_document('returns')}
print('returns#00 now says:', stored['returns#00'].text[:60], '...')

content_changed | updated = 1 | unchanged = 2 | embedded = 1
returns#00 now says: We accept returns within 45 days of delivery. Refunds are is ...


## 4. Shrink the document: prune stale chunks

The classic bug is upserting the new chunks and leaving the old tail behind. Those orphans can outrank the fresh text, and citations point at deleted content.

In [5]:
shorter = Document('returns', 'Return Policy', edited.text.split(chr(10) * 2)[0])
report = pipeline.ingest(shorter)
print(report.reason, '| deleted =', report.deleted, '| rows left =',
      len(pipeline.store.rows_for_document('returns')))
assert report.deleted >= 1
assert len(pipeline.store.rows_for_document('returns')) == 1

content_changed | deleted = 2 | rows left = 1


## 5. Batch embedding: requests, not texts

Nine chunks with `batch_size=4` -> 3 requests. At thousands of chunks this is the difference between a few requests and a few thousand.

In [6]:
long_doc = Document(
    'batch', 'Batch demo',
    chr(10).join(f'Paragraph {i} about returns and refunds within 30 days of delivery.' for i in range(9)),
)
batch_pipeline = Pipeline(max_chars=80, batch_size=4)
report = batch_pipeline.ingest(long_doc)
print('chunks =', report.chunks_total, '| batch requests =', report.embed_batches,
      '| texts embedded =', report.embedded)
assert report.chunks_total == 9
assert report.embed_batches == 3

chunks = 9 | batch requests = 3 | texts embedded = 9


## 6. Re-index triggers: profile change

Changing the chunker version (or the model/dimensions) invalidates every stored vector for the document. `reindex_reason` makes that explicit instead of leaving two incompatible profiles in one index.

In [7]:
profile_pipeline = Pipeline(max_chars=200)
profile_pipeline.ingest(sample['returns'])
print('fresh document ->', profile_pipeline.reindex_reason(sample['returns']))
profile_pipeline.chunker_version = 'paragraph-v2'
print('after chunker change ->', profile_pipeline.reindex_reason(sample['returns']))
report = profile_pipeline.ingest(sample['returns'])
print('rebuild:', report.reason, '| embedded =', report.embedded, '| deleted =', report.deleted)
assert report.reason == 'profile_changed'
assert report.embedded == report.chunks_total

fresh document -> None
after chunker change -> profile_changed
rebuild: profile_changed | embedded = 3 | deleted = 3


## 7. Query -> retrieve -> assemble -> answer

A clean run: the gold chunk is first, the answer quotes it, and the citation marker came from assembly.

In [8]:
search_pipeline = Pipeline(max_chars=200, batch_size=4)
for document in SAMPLE_DOCUMENTS:
    search_pipeline.ingest(document)

question = 'How many days do I have to return an item?'
retrieval, assembled, answer = search_pipeline.query(question)
print('hits:', [(row.chunk_id, round(score, 4)) for row, score in retrieval.hits])
print('prompt chars used:', assembled.used_chars, '/', assembled.budget_chars)
print('answer:', answer.text)
print('citations:', answer.citations, '| abstained:', answer.abstained)
assert retrieval.hits[0][0].doc_id == 'returns'
assert not answer.abstained

hits: [('returns#00', 0.147), ('returns#02', 0.147), ('shipping#00', 0.0803)]
prompt chars used: 443 / 700
answer: We accept returns within 30 days of delivery. [1]
citations: [1] | abstained: False


## 8. The done-when artifact: trace one document

Every stage reports the numbers it produced. `missing 1` is not an error: k=3 but this pipeline only ingested this one document, so only two rows exist. That is the `k is a cap, not a promise` failure in miniature.

In [9]:
trace_pipeline = Pipeline(max_chars=200, batch_size=4)
trace = trace_pipeline.trace_document(
    sample['warranty'], 'How long do manufacturer warranties cover defects?'
)
print(trace.format())
assert [stage.stage for stage in trace.stages] == list(STAGES)
assert not trace.answer.abstained

trace: warranty -> 'How long do manufacturer warranties cover defects?'
  ingest    new_document: 2 embedded, 0 unchanged, 0 pruned
            warnings: ['none']
  chunk     2 chunks, max 112 chars
            chunk_ids: ['warranty#00', 'warranty#01']
  embed     2 rows x 8192 dims, 0 zero vectors
            batches: 1
            model: hashing-8192-v1
  store     2 rows total; profile hashing-8192-v1|8192|paragraph-v1|200
            chunks_for_document: 2
  query     query vector norm 1.000
            zero_query: False
  retrieve  2 hits for k=3 (missing 1)
            hits: [('warranty#00', 0.4865), ('warranty#01', 0.0)]
  assemble  228/700 chars, 2 citations
            dropped: ['none']
  answer    quoted 1 citation(s)
            text: Manufacturer warranties cover defects for 12 months from the purchase date. [1]
            citations: [1]


## 9. A silent failure you can reproduce: morphology

Ask `How long is the warranty?`. The chunk with the answer says `warranties`, not `warranty`; the hashing embedder misses it. Retrieval returns a different warranty chunk, and the extractive answerer quotes it with a *valid* citation. No exception, plausible output, wrong answer.

In [10]:
morph_pipeline = Pipeline(max_chars=200, batch_size=4)
for document in SAMPLE_DOCUMENTS:
    morph_pipeline.ingest(document)

retrieval, assembled, answer = morph_pipeline.query('How long is the warranty?')
print('hits:', [(row.chunk_id, round(score, 4)) for row, score in retrieval.hits])
print('answer:', answer.text)
assert 'warranty#00' not in [row.chunk_id for row, _ in retrieval.hits]
assert answer.text.startswith('Warranty claims')

hits: [('warranty#01', 0.14), ('returns#00', 0.0), ('returns#01', 0.0)]
answer: Warranty claims require the original receipt and the serial number printed on the device. [1]


## 10. Diagnostics: turn silent conditions into warnings

Empty query, k larger than the store, and a budget too small to include anything. Each is a stage-level issue with a code and a message.

In [11]:
issues = search_pipeline.diagnose(
    document=sample['returns'], query='', k=100, budget_chars=10
)
for issue in issues:
    print(f'{issue.stage:<9} {issue.code:<14} {issue.message}')
codes = {issue.code for issue in issues}
assert {'zero_query', 'short_k', 'budget_drop', 'abstained'} <= codes

query     zero_query     query embedded to a zero vector: no result can match
retrieve  short_k        retrieved 7 rows for k=100
assemble  budget_drop    7 chunk(s) dropped by the context budget
answer    abstained      answerer found no supporting sentence (abstained)


## 11. The silent-failure catalogue, stage by stage

Two to three concrete ways each stage can break without raising. This is the checklist to read before blaming the model.

In [12]:
for stage in STAGES:
    print(stage)
    for note in silent_failure_notes(stage):
        print('  -', note)
    print()

ingest
  - an unsupported/corrupt file yields zero documents and the job still reports success
  - a duplicate doc_id overwrites (or is skipped by) the previous version without a diff
  - an HTML/PDF loader returns navigation chrome as if it were content

chunk
  - a document shorter than the chunker window produces one giant chunk that the embedder truncates
  - the last chunk is a 30-character stub whose embedding matches everything weakly
  - tables and code fences are sliced mid-structure - retrieval still returns something

embed
  - input beyond the model window is truncated by the provider without an error
  - empty/OOV text becomes a zero vector: stored fine, cosine=NaN, never retrieved
  - a retried batch silently drops the rows that failed instead of failing the job

store
  - stale chunks from the previous document version remain and can outrank fresh ones
  - batched writes partially succeed and the ingest counter still says 'done'
  - a dimension mismatch is coerced by an 

## 12. Run the tests

In [13]:
import subprocess
import sys

result = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(result.stdout[-1800:])
assert result.returncode == 0, "pytest failures!"
assert "49 passed" in result.stdout, "unexpected test count"
print("offline tests passed")


.................................................                        [100%]
49 passed in 0.40s

offline tests passed


## 13. What to notice

- re-ingesting an unchanged document costs **zero** embedding calls
- one edited paragraph re-embeds exactly one chunk; shrinking prunes the tail
- batching counts requests: 9 chunks / batch 4 = 3 calls
- a chunker/model/dimension change is a `profile_changed` full rebuild
- `k > rows` returns fewer hits and reports `missing`
- a wrong answer with a valid citation can still be a **retrieval** failure
- every stage needs a detector, not just a description